# Drop the imaging wells with poor quality

In [32]:
import polars as pl
import os
import numpy as np

IMG_QC_DIR = "../../0_data_prep/inputs/varchamp_img_well_qc"
BINS = 50
IMG_QC_OUT = "../../0_data_prep/outputs/varchamp_img_qc_res"

## Only DAPI and GFP channels matter for Cytoself
channel_list = ["DAPI", "GFP"]
os.listdir(IMG_QC_DIR)

['2025_05_23_Batch_17',
 '2025_06_10_Batch_19',
 '2024_01_23_Batch_7',
 '2024_12_09_Batch_11',
 '2025_03_17_Batch_16',
 '2024_12_09_Batch_12',
 '2024_02_06_Batch_8',
 '2025_06_10_Batch_18',
 '2025_03_17_Batch_15',
 '2025_01_27_Batch_13',
 '2025_01_28_Batch_14']

## 1. Collecting Imaging Well QC Metrics

In [42]:
def get_plate_well_sum(batch, img_qc_dir, output_dir, bins=BINS, write_pq=False):
    plate_sum = pl.read_parquet(f"{img_qc_dir}/{batch}/plate_sum_stats.parquet")
    plate_well_sum = pl.read_parquet(f"{img_qc_dir}/{batch}/plate_well_sum_stats.parquet")
    # plate_site_sum = pl.read_parquet(f"../outputs/1.plate_bg_summary/{batch}/plate_site_channel.parquet")
    # plate_site_sum = plate_site_sum.filter(~(pl.col("channel").str.contains("Brightfield"))).sort(by=["plate","well"]).join(plate_sum, on=["plate","channel"], how="left", suffix="_plate")

    # plate_well_sum.filter(~(pl.col("channel").str.contains("Brightfield"))).sort(by=["plate","well"])
    plate_well_sum = plate_well_sum.filter(
        ~(pl.col("channel").str.contains("Brightfield"))
    ).join(
        plate_sum.rename({"perc_50": "median_plate"}), on=["plate","channel"], how="left", suffix="_plate"
    ).sort(by=["plate", "well"])

    plates = sorted(plate_well_sum["plate"].unique())
    plate_well_channel_qc_results, plate_well_qc_results = pl.DataFrame(), pl.DataFrame()
    for plate in plates: #plate_well_sum.group_by("plate"):
        group = plate_well_sum.filter(pl.col("plate")==plate)
        # plate = group[0][0]
        # fig, axes = plt.subplots(1, 4, figsize=(15, 3))
        plate_well_channel_qc_results_plate = pl.DataFrame()
        for channel in channel_list:
            # print(plate, channel)
            plate_channel_plate = plate_well_sum.filter(
                (pl.col("plate")==plate)&(pl.col("channel")==channel)
            ).with_columns(
                (pl.col("perc_99") / pl.col("perc_25")).alias("s2n_ratio")
            ).filter(
                (~(pl.col("s2n_ratio").is_infinite()))&(~(pl.col("s2n_ratio").is_nan()))
            ).sort(by="s2n_ratio")
            if channel == "GFP":
                plate_channel_plate = plate_channel_plate.with_columns(
                    np.log10(pl.col("s2n_ratio")+1e-4).alias("s2n_ratio")
                )
            s2n_ratios = plate_channel_plate["s2n_ratio"].to_numpy()
            counts, edges = np.histogram(s2n_ratios, bins=bins)
            # 4) the right edge of the first bin is our cutoff
            threshold = edges[1]
            plate_channel_plate = plate_channel_plate.with_columns(
                pl.lit(threshold).alias("s2n_threshold"),
                (pl.col("s2n_ratio") <= threshold).alias("is_bg")
            )
            
            # sns.histplot(plate_channel_plate["s2n_ratio"], ax=axes[channel_list.index(channel)], bins=100) # log_scale=True
            # axes[channel_list.index(channel)].axvline(threshold, color='red', linestyle='--', label='Threshold')
            # if channel == "GFP":
            #     axes[channel_list.index(channel)].set_xlabel("log10(s2n_ratio)")
            # else:
            #     axes[channel_list.index(channel)].set_xlabel("s2n_ratio")
            # axes[channel_list.index(channel)].set_title(channel)
            plate_well_channel_qc_results_plate = pl.concat([plate_well_channel_qc_results_plate, plate_channel_plate])
            plate_well_channel_qc_results = pl.concat([plate_well_channel_qc_results, plate_channel_plate])

        # fig.suptitle(f"{plate}",y=1.05)
        # fig.subplots_adjust(wspace=0.25)

        plate_well_sum_with_metrics_agg = plate_well_channel_qc_results_plate.group_by(
            ["plate", "well"]
        ).agg(
            pl.col("is_bg").max().alias("is_bg")
        )
        # display(plate_well_sum_with_metrics_agg.unique(subset=["plate", "well"]).shape)
        plate_well_qc_results = pl.concat([plate_well_qc_results, plate_well_sum_with_metrics_agg])
        # display(plate_well_qc_results.shape)
        
    output_pq_file, out_pl_well_qc = f"{output_dir}/{batch}/plate_well_channel_sum_with_qc_metrics.parquet", f"{output_dir}/{batch}/plate_well_sum_with_qc_metrics.parquet"        
    if write_pq:
        os.makedirs(f"{output_dir}/{batch}", exist_ok=True)
        print("Writed outputs at:", output_pq_file, out_pl_well_qc)
        plate_well_channel_qc_results.write_parquet(
            output_pq_file
        )
        plate_well_qc_results.write_parquet(
            out_pl_well_qc
        )
        
    return plate_well_sum_with_metrics, plate_well_qc_results

In [43]:
bio_rep_batches = os.listdir(IMG_QC_DIR)

for batch in bio_rep_batches:
    output_dir = f"{IMG_QC_OUT}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    plate_well_sum_with_metrics = get_plate_well_sum(batch, IMG_QC_DIR, output_dir, write_pq=True)
    # display(plate_well_sum_with_metrics.head())

Writed outputs at: ../../0_data_prep/outputs/varchamp_img_qc_res/2025_05_23_Batch_17/plate_well_channel_sum_with_qc_metrics.parquet ../../0_data_prep/outputs/varchamp_img_qc_res/2025_05_23_Batch_17/plate_well_sum_with_qc_metrics.parquet
Writed outputs at: ../../0_data_prep/outputs/varchamp_img_qc_res/2025_06_10_Batch_19/plate_well_channel_sum_with_qc_metrics.parquet ../../0_data_prep/outputs/varchamp_img_qc_res/2025_06_10_Batch_19/plate_well_sum_with_qc_metrics.parquet
Writed outputs at: ../../0_data_prep/outputs/varchamp_img_qc_res/2024_01_23_Batch_7/plate_well_channel_sum_with_qc_metrics.parquet ../../0_data_prep/outputs/varchamp_img_qc_res/2024_01_23_Batch_7/plate_well_sum_with_qc_metrics.parquet
Writed outputs at: ../../0_data_prep/outputs/varchamp_img_qc_res/2024_12_09_Batch_11/plate_well_channel_sum_with_qc_metrics.parquet ../../0_data_prep/outputs/varchamp_img_qc_res/2024_12_09_Batch_11/plate_well_sum_with_qc_metrics.parquet
Writed outputs at: ../../0_data_prep/outputs/varchamp_

## 2. Cropping cell images

In [44]:
"""Filter cells and crop image.
For each allele:
1. Filter cells for QA/QC
2. Extract 100x100 pixel crop for gfp and dapi
3. Save as stacked numpy array
"""  # noqa: INP001

import os
import glob
# import argparse
import numpy as np
import polars as pl
import zarr
from tqdm import tqdm

VARCHAMP_CP_DIR = "../../0_data_prep/inputs/varchamp_cellpainting_gallery"
ZARR_IMG_DIR = f"../../0_data_prep/outputs/zarr_images"
VARCHAMP_PROF_DIR = "../../0_data_prep/inputs/varchamp_snakemake_batch_profiles"
VARCHAMP_IMG_WELL_QC_DIR = "../../0_data_prep/outputs/varchamp_img_qc_res"


def crop_allele(allele: str, profile_df: pl.DataFrame, out_dir: str) -> None:
    """Crop images and save metadata as numpy arrays for one allele.

    Parameters
    ----------
    allele : String
        Name of allele to process
    profile_df : String
        Dataframe with pathname and cell coordinates
    img_dir : String
        Directory where all images are stored
    out_dir : String
        Directory where numpy arrays should be saved

    """
    allele_df = profile_df.filter(pl.col("Protein_label") == allele)
    sites = allele_df.select("Metadata_SiteID").to_series().unique().to_list()

    meta = []
    gfp = []
    dna = []

    for site in sites:
        site_df = allele_df.filter(pl.col("Metadata_SiteID") == site)

        meta.append(site_df.select([
            "Protein_label",
            "Metadata_CellID",
        ]))

        dna_path = site_df[["DNA_zarrpath"]][0].item()
        gfp_path = site_df[["GFP_zarrpath"]][0].item()
        # gfp_path = f"{img_dir}/{gfp_zarr}"
        # dna_path = f"{img_dir}/{dna_zarr}"

        gfp_img = zarr.open(gfp_path)
        dna_img = zarr.open(dna_path)
        for row in site_df.iter_rows(named=True):
            x1, x2 = row["x_low"], row["x_high"]
            y1, y2 = row["y_low"], row["y_high"]

            ## Note, need to flip the coordinates!!!
            gfp.append(gfp_img[y1:y2, x1:x2])
            dna.append(dna_img[y1:y2, x1:x2])

    # Stack and save arrays
    gfp_array = np.stack(gfp)
    dna_array = np.stack(dna)
    meta_array = pl.concat(meta).to_numpy()

    np.save(f"{out_dir}/{allele}_label.npy", meta_array)
    np.save(f"{out_dir}/{allele}_pro.npy", gfp_array)
    np.save(f"{out_dir}/{allele}_nuc.npy", dna_array)


def filter_cell_crop(batch_ids):
    """Filter cells and crop data.

    Filter all cells according to many QA/QC criteria and then crop cells.

    """
    # parser = argparse.ArgumentParser(description="Convert TIFF images to Zarr format.")
    # parser.add_argument("--batch_ids", required=True, help="Batch ID for processing images.")
    # parser.add_argument("--n_thread", type=int, default=128, help="Number of threads to use.")
    # args = parser.parse_args()
    # batch_ids = args.batch_ids

    for batch_id in batch_ids.split(","):
        print("Processing batch ID: ", batch_id)
        
        imagecsv_dir = f"{VARCHAMP_CP_DIR}/{batch_id}"
        prof_path = f"{VARCHAMP_PROF_DIR}/{batch_id}/profiles.parquet"
        img_well_qc = f"{VARCHAMP_IMG_WELL_QC_DIR}/{batch_id}/plate_well_sum_with_qc_metrics.parquet"

        # Filter thresholds
        min_area_ratio = 0.15
        max_area_ratio = 0.3
        min_center = 50
        max_center = 1030
        num_mad = 5
        min_cells = 250
        
        # Get metadata
        profiles = pl.scan_parquet(prof_path).select(
            ["Metadata_well_position", "Metadata_plate_map_name", "Metadata_ImageNumber", "Metadata_ObjectNumber",
            "Metadata_symbol", "Metadata_gene_allele", "Metadata_node_type", "Metadata_Plate",
            "Nuclei_AreaShape_Area", "Cells_AreaShape_Area", "Nuclei_AreaShape_Center_X", "Nuclei_AreaShape_Center_Y",
            "Cells_Intensity_MeanIntensity_GFP", "Cells_Intensity_MedianIntensity_GFP", "Cells_Intensity_IntegratedIntensity_GFP"],
        ).collect()

        # Filter based on cell to nucleus area
        profiles = profiles.with_columns(
                        (pl.col("Nuclei_AreaShape_Area")/pl.col("Cells_AreaShape_Area")).alias("Nucleus_Cell_Area"),
                        pl.concat_str([
                            "Metadata_Plate", "Metadata_well_position", "Metadata_ImageNumber", "Metadata_ObjectNumber",
                            ], separator="_").alias("Metadata_CellID"),
                ).filter((pl.col("Nucleus_Cell_Area") > min_area_ratio) & (pl.col("Nucleus_Cell_Area") < max_area_ratio))

        # Filter cells too close to image edge
        profiles = profiles.filter(
            ((pl.col("Nuclei_AreaShape_Center_X") > min_center) & (pl.col("Nuclei_AreaShape_Center_X") < max_center) &
            (pl.col("Nuclei_AreaShape_Center_Y") > min_center) & (pl.col("Nuclei_AreaShape_Center_Y") < max_center)),
        )

        # Calculate mean, median and mad of gfp intensity for each allele
        ## mean
        means = profiles.group_by(["Metadata_Plate", "Metadata_well_position"]).agg(
            pl.col("Cells_Intensity_MeanIntensity_GFP").mean().alias("WellIntensityMean"),
        )
        profiles = profiles.join(means, on=["Metadata_Plate", "Metadata_well_position"])
        ## median
        medians = profiles.group_by(["Metadata_Plate", "Metadata_well_position"]).agg(
            pl.col("Cells_Intensity_MedianIntensity_GFP").median().alias("WellIntensityMedian"),
        )
        profiles = profiles.join(medians, on=["Metadata_Plate", "Metadata_well_position"])
        ## mad
        profiles = profiles.with_columns(
            (pl.col("Cells_Intensity_MedianIntensity_GFP") - pl.col("WellIntensityMedian")).abs().alias("Abs_dev"),
        )
        mad = profiles.group_by(["Metadata_Plate", "Metadata_well_position"]).agg(
            pl.col("Abs_dev").median().alias("Intensity_MAD"),
        )
        profiles = profiles.join(mad, on=["Metadata_Plate", "Metadata_well_position"])

        # Threshold is 5X
        ## Used to be median well intensity + 5*mad implemented by Jess
        ## Proposing to mean well intensity + 5*mad implemented by Runxi, still testing
        profiles = profiles.with_columns(
            (pl.col("WellIntensityMedian") + num_mad*pl.col("Intensity_MAD")).alias("Intensity_upper_threshold"), ## pl.col("WellIntensityMedian"), pl.col("WellIntensityMean")
            (pl.col("WellIntensityMedian") - num_mad*pl.col("Intensity_MAD")).alias("Intensity_lower_threshold"), ## pl.col("WellIntensityMedian"), pl.col("WellIntensityMean")
        )
        ## Filter by intensity MAD
        profiles = profiles.filter(
            pl.col("Cells_Intensity_MeanIntensity_GFP") <= pl.col("Intensity_upper_threshold"),
        ).filter(
            pl.col("Cells_Intensity_MeanIntensity_GFP") >= pl.col("Intensity_lower_threshold"),
        )

        # Filter out alleles with fewer than 250 cells
        keep_alleles = profiles.group_by("Metadata_gene_allele").len().filter(
            pl.col("len") >= min_cells,
            ).select("Metadata_gene_allele").to_series().to_list()
        profiles = profiles.filter(pl.col("Metadata_gene_allele").is_in(keep_alleles))

        # add full crop coordinates
        profiles = profiles.with_columns(
            (pl.col("Nuclei_AreaShape_Center_X") - 50).alias("x_low").round().cast(pl.Int16),
            (pl.col("Nuclei_AreaShape_Center_X") + 50).alias("x_high").round().cast(pl.Int16),
            (pl.col("Nuclei_AreaShape_Center_Y") - 50).alias("y_low").round().cast(pl.Int16),
            (pl.col("Nuclei_AreaShape_Center_Y") + 50).alias("y_high").round().cast(pl.Int16),
        )

        ## add img well QC results
        img_qc_df = pl.read_parquet(img_well_qc)
        profiles = profiles.join(
            img_qc_df,
            left_on=["Metadata_Plate", "Metadata_well_position"],
            right_on=["plate", "well"],
            how="left"
        ).filter(
            ~pl.col("is_bg")
        )

        # Read in all Image.csv to get ImageNumber:SiteNumber mapping and paths
        image_dat = []
        icfs = glob.glob(os.path.join(imagecsv_dir, "**/*Image.csv"), recursive=True)
        for icf in tqdm(icfs):
            fp = icf.split('/')[-2]
            plate, well = "-".join(fp.split("-")[:-2]), fp.split("-")[-2]
            image_dat.append(pl.read_csv(icf).select(
                [
                    "ImageNumber",
                    "Metadata_Site",
                    "PathName_OrigDNA",
                    "FileName_OrigDNA",
                    "FileName_OrigGFP",
                    ],
                ).with_columns(
                    pl.lit(plate).alias("Metadata_Plate"),
                    pl.lit(well).alias("Metadata_well_position"),
                )
            )

        image_dat = pl.concat(image_dat).rename({"ImageNumber": "Metadata_ImageNumber"})

        # Create useful filepaths
        image_dat = image_dat.with_columns(
            pl.col("PathName_OrigDNA").str.replace(".*cpg0020-varchamp/broad/images", VARCHAMP_CP_DIR).alias("Path_root"),
        )
        image_dat = image_dat.with_columns(
            pl.concat_str(["Path_root", "FileName_OrigDNA"], separator="/").str.replace_many(
                [VARCHAMP_CP_DIR, "tiff"], [ZARR_IMG_DIR, "zarr"]).alias("DNA_zarrpath"),
            pl.concat_str(["Path_root", "FileName_OrigGFP"], separator="/").str.replace_many(
                [VARCHAMP_CP_DIR, "tiff"], [ZARR_IMG_DIR, "zarr"]).alias("GFP_zarrpath"),
        )

        image_dat = image_dat.drop([
            "PathName_OrigDNA",
            "FileName_OrigDNA",
            "FileName_OrigGFP",
            "Path_root",
        ])

        # Append to profiles
        profiles = profiles.join(image_dat, on = ["Metadata_Plate", "Metadata_well_position", "Metadata_ImageNumber"])

        # Sort by allele, then image number
        profiles = profiles.with_columns(
            pl.concat_str(["Metadata_Plate", "Metadata_well_position", "Metadata_Site"], separator="_").alias(
                "Metadata_SiteID"),
            pl.col("Metadata_gene_allele").str.replace("_", "-").alias("Protein_label"),
        )
        profiles = profiles.sort(["Protein_label", "Metadata_SiteID"])
        alleles = profiles.select("Protein_label").to_series().unique().to_list()

        ## Create output directory
        out_dir = f"../inputs/1_model_input/{batch_id}"
        os.makedirs(out_dir, exist_ok=True)
        for allele in tqdm(alleles):
            crop_allele(allele, profiles, out_dir)


filter_cell_crop("2025_03_17_Batch_15")

Processing batch ID:  2025_03_17_Batch_15


plate,well,is_bg
str,str,bool
"""2025-03-17_B15A1A2_P1T1""","""M12""",true
"""2025-03-17_B15A1A2_P1T1""","""F05""",true
"""2025-03-17_B15A1A2_P1T1""","""K01""",true
"""2025-03-17_B15A1A2_P1T1""","""N21""",false
"""2025-03-17_B15A1A2_P1T1""","""P21""",false
…,…,…
"""2025-03-17_B15A1A2_P1T4""","""A23""",false
"""2025-03-17_B15A1A2_P1T4""","""C22""",false
"""2025-03-17_B15A1A2_P1T4""","""O19""",false


plate,well,is_bg
str,str,bool
"""2025-03-17_B15A1A2_P1T1""","""G22""",false
"""2025-03-17_B15A1A2_P1T2""","""D15""",false
"""2025-03-17_B15A1A2_P1T4""","""O12""",false
"""2025-03-17_B15A1A2_P1T4""","""K23""",false
"""2025-03-17_B15A1A2_P1T1""","""B05""",false
…,…,…
"""2025-03-17_B15A1A2_P1T4""","""I19""",false
"""2025-03-17_B15A1A2_P1T2""","""M18""",false
"""2025-03-17_B15A1A2_P1T4""","""G20""",false


Metadata_well_position,Metadata_plate_map_name,Metadata_ImageNumber,Metadata_ObjectNumber,Metadata_symbol,Metadata_gene_allele,Metadata_node_type,Metadata_Plate,Nuclei_AreaShape_Area,Cells_AreaShape_Area,Nuclei_AreaShape_Center_X,Nuclei_AreaShape_Center_Y,Cells_Intensity_MeanIntensity_GFP,Cells_Intensity_MedianIntensity_GFP,Cells_Intensity_IntegratedIntensity_GFP,Nucleus_Cell_Area,Metadata_CellID,WellIntensityMean,WellIntensityMedian,Abs_dev,Intensity_MAD,Intensity_upper_threshold,Intensity_lower_threshold,x_low,x_high,y_low,y_high
str,str,i64,i64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,i16,i16,i16,i16
"""A01""","""B15A1A2_P1""",4,2,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1442,7482,584.278779,69.434119,0.002032,0.002009,15.201082,0.192729,"""2025-03-17_B15A1A2_P1T4_A01_4_…",0.004345,0.003418,0.001408,0.000963,0.008231,-0.001395,534,634,19,119
"""A01""","""B15A1A2_P1""",1,2,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1777,10231,591.697243,403.916151,0.002078,0.001867,21.263537,0.173688,"""2025-03-17_B15A1A2_P1T4_A01_1_…",0.004345,0.003418,0.001551,0.000963,0.008231,-0.001395,542,642,354,454
"""A01""","""B15A1A2_P1""",7,3,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1366,8060,261.579063,76.528551,0.00246,0.002453,19.830402,0.169479,"""2025-03-17_B15A1A2_P1T4_A01_7_…",0.004345,0.003418,0.000965,0.000963,0.008231,-0.001395,212,312,27,127
"""A01""","""B15A1A2_P1""",1,3,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1232,4359,252.521916,417.189123,0.0046,0.004946,20.053236,0.282634,"""2025-03-17_B15A1A2_P1T4_A01_1_…",0.004345,0.003418,0.001528,0.000963,0.008231,-0.001395,203,303,367,467
"""A01""","""B15A1A2_P1""",3,4,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1629,10590,241.637201,104.881522,0.006784,0.007277,71.842105,0.153824,"""2025-03-17_B15A1A2_P1T4_A01_3_…",0.004345,0.003418,0.003859,0.000963,0.008231,-0.001395,192,292,55,155
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""F20""","""B15A1A2_P1""",1259,108,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1480,7151,561.129054,908.225676,0.005086,0.004677,36.368077,0.206964,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.001356,0.001118,0.008913,-0.00227,511,611,858,958
"""F20""","""B15A1A2_P1""",1259,109,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1560,9556,644.808333,931.039103,0.005042,0.004345,48.18109,0.163248,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.001024,0.001118,0.008913,-0.00227,595,695,881,981
"""F20""","""B15A1A2_P1""",1259,112,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1314,8079,228.368341,993.426941,0.003481,0.003337,28.125522,0.162644,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.000016,0.001118,0.008913,-0.00227,178,278,943,1043


Metadata_well_position,Metadata_plate_map_name,Metadata_ImageNumber,Metadata_ObjectNumber,Metadata_symbol,Metadata_gene_allele,Metadata_node_type,Metadata_Plate,Nuclei_AreaShape_Area,Cells_AreaShape_Area,Nuclei_AreaShape_Center_X,Nuclei_AreaShape_Center_Y,Cells_Intensity_MeanIntensity_GFP,Cells_Intensity_MedianIntensity_GFP,Cells_Intensity_IntegratedIntensity_GFP,Nucleus_Cell_Area,Metadata_CellID,WellIntensityMean,WellIntensityMedian,Abs_dev,Intensity_MAD,Intensity_upper_threshold,Intensity_lower_threshold,x_low,x_high,y_low,y_high,is_bg
str,str,i64,i64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,i16,i16,i16,i16,bool
"""A01""","""B15A1A2_P1""",4,2,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1442,7482,584.278779,69.434119,0.002032,0.002009,15.201082,0.192729,"""2025-03-17_B15A1A2_P1T4_A01_4_…",0.004345,0.003418,0.001408,0.000963,0.008231,-0.001395,534,634,19,119,false
"""A01""","""B15A1A2_P1""",1,2,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1777,10231,591.697243,403.916151,0.002078,0.001867,21.263537,0.173688,"""2025-03-17_B15A1A2_P1T4_A01_1_…",0.004345,0.003418,0.001551,0.000963,0.008231,-0.001395,542,642,354,454,false
"""A01""","""B15A1A2_P1""",7,3,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1366,8060,261.579063,76.528551,0.00246,0.002453,19.830402,0.169479,"""2025-03-17_B15A1A2_P1T4_A01_7_…",0.004345,0.003418,0.000965,0.000963,0.008231,-0.001395,212,312,27,127,false
"""A01""","""B15A1A2_P1""",1,3,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1232,4359,252.521916,417.189123,0.0046,0.004946,20.053236,0.282634,"""2025-03-17_B15A1A2_P1T4_A01_1_…",0.004345,0.003418,0.001528,0.000963,0.008231,-0.001395,203,303,367,467,false
"""A01""","""B15A1A2_P1""",3,4,"""AP2S1""","""AP2S1""","""disease_wt""","""2025-03-17_B15A1A2_P1T4""",1629,10590,241.637201,104.881522,0.006784,0.007277,71.842105,0.153824,"""2025-03-17_B15A1A2_P1T4_A01_3_…",0.004345,0.003418,0.003859,0.000963,0.008231,-0.001395,192,292,55,155,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""F20""","""B15A1A2_P1""",1259,108,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1480,7151,561.129054,908.225676,0.005086,0.004677,36.368077,0.206964,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.001356,0.001118,0.008913,-0.00227,511,611,858,958,false
"""F20""","""B15A1A2_P1""",1259,109,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1560,9556,644.808333,931.039103,0.005042,0.004345,48.18109,0.163248,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.001024,0.001118,0.008913,-0.00227,595,695,881,981,false
"""F20""","""B15A1A2_P1""",1259,112,"""PRKACB""","""PRKACB""","""NC""","""2025-03-17_B15A1A2_P1T3""",1314,8079,228.368341,993.426941,0.003481,0.003337,28.125522,0.162644,"""2025-03-17_B15A1A2_P1T3_F20_12…",0.011266,0.003322,0.000016,0.001118,0.008913,-0.00227,178,278,943,1043,false
